# **Chapter 4: Privacy Attacks and Information Extraction Lab** 🔒🕵️

---

## **Lab Overview**

Welcome to the Privacy Attacks lab! This hands-on exercise teaches how attackers can **extract sensitive information** from trained machine learning models—even when they only have black-box query access (no source code, no training data).

### **What You'll Learn**
✅ **Membership Inference**: Determine if a specific person's data was in the training set  
✅ **Model Inversion**: Reconstruct training data features from model behavior  
✅ **Property Inference**: Discover hidden dataset properties (demographics, distributions)  
✅ **Model Extraction**: Steal model functionality via API queries  
✅ **Privacy Defenses**: Output perturbation, differential privacy, rate limiting

### **Security Context**
Privacy attacks threaten real-world deployed systems:
- **Healthcare**: Determine if someone's medical record was used to train a diagnostic model
- **Finance**: Infer customer demographics from loan approval models
- **Face Recognition**: Reconstruct facial features from model outputs
- **Proprietary Models**: Steal commercial ML models (e.g., GPT, fraud detection systems)
- **GDPR/Privacy Laws**: Membership inference reveals compliance violations (data that should have been deleted)

### **Prerequisites**
- Python basics (variables, loops, functions)
- Understanding of neural networks (from Chapters 1-3)
- Familiarity with overfitting and model confidence
- Basic privacy concepts (what is "sensitive data"?)

### **Lab Structure**
1. **Membership Inference**: Distinguish training vs non-training samples using confidence patterns
2. **Model Inversion**: Reconstruct representative class features via gradient ascent
3. **Property Inference**: Infer hidden dataset properties from model behavior
4. **Model Extraction**: Build surrogate model by querying victim API
5. **Defenses**: Output noise, top-k filtering, differential privacy concepts

**Estimated Time:** 60-75 minutes

### **Important Note: Ethical Considerations**
These techniques are taught for **defensive purposes**:
- Understand privacy risks when deploying ML systems
- Conduct privacy audits before production deployment
- Design privacy-preserving ML architectures
- Never use these attacks on systems without authorization

---

---

## **Step 1: Environment Setup & Data Loading** 📦

### **What We're Doing**
Preparing a **carefully partitioned** dataset to simulate privacy attack scenarios. We need to create distinct groups:
- **IN samples**: Data used during model training (training + test sets)
- **OUT samples**: Data the model has never seen (holdout set)

### **Why This Matters**
Privacy attacks exploit the difference between how models treat IN vs OUT samples:
- **Overfitted models** memorize training data → higher confidence on IN samples
- **OUT samples** are generalized → lower confidence, higher uncertainty
- This confidence gap enables membership inference attacks

### **Key Concept: Training Data Privacy**
When you train a model, the training data **leaves traces** in the model's parameters. Privacy attacks exploit these traces to infer:
- Which individuals' data was used (membership)
- What the data looked like (inversion)
- Dataset properties (demographics, distributions)

### **What to Expect**
- Load MNIST dataset
- Partition into IN (60% train, 20% test) and OUT (20% holdout)
- Use smaller subsets for lab speed

In [ ]:
# Import required libraries for privacy attack experiments
import numpy as np                          # Numerical operations
import pandas as pd                         # Data manipulation
import matplotlib.pyplot as plt             # Visualization
import seaborn as sns                       # Statistical plots
from sklearn.datasets import fetch_openml  # MNIST dataset
from sklearn.model_selection import train_test_split  # Data splitting
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report  # Evaluation
from sklearn.neural_network import MLPClassifier  # Neural network
from sklearn.ensemble import RandomForestClassifier  # For attack model
import warnings
warnings.filterwarnings('ignore')  # Suppress warnings for cleaner output

print("=" * 50)
print("ENVIRONMENT CHECK")
print("=" * 50)
print("✓ All imports successful!")
print("✓ NumPy, Pandas, Matplotlib, Scikit-learn ready")
print("=" * 50)

# Set visualization style and random seed for reproducibility
sns.set(style="whitegrid")
np.random.seed(42)

In [ ]:
# Load MNIST dataset with CAREFUL partitioning for privacy experiments
print("Loading MNIST dataset...")
print("(This may take 10-20 seconds on first run...)")

mnist = fetch_openml('mnist_784', version=1, parser='auto')
# Normalize pixel values to [0, 1] range
X_full = mnist.data.astype('float32') / 255.0
y_full = mnist.target.astype('int')

# CRITICAL: Create distinct populations for IN (training) and OUT (holdout)
# This separation simulates real-world privacy attack scenarios
np.random.seed(42)

print("\n" + "=" * 50)
print("DATA PARTITIONING FOR PRIVACY EXPERIMENTS")
print("=" * 50)

# Step 1: Split into 80% IN distribution, 20% OUT (holdout - never seen)
X_temp, X_holdout, y_temp, y_holdout = train_test_split(
    X_full, y_full, 
    test_size=0.2,      # 20% completely held out (OUT samples)
    random_state=42, 
    stratify=y_full     # Maintain class balance
)

# Step 2: Further split IN distribution into train (60%) and test (20%)
X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(
    X_temp, y_temp, 
    test_size=0.25,     # 0.25 * 0.8 = 0.2 (20% of total)
    random_state=42, 
    stratify=y_temp
)

# Step 3: Use smaller subsets for faster lab experimentation
train_size = 5000   # Enough to train but fast
test_size = 1000    # IN samples (for membership inference)
holdout_size = 1000 # OUT samples (for membership inference)

X_train = X_train_full.iloc[:train_size].reset_index(drop=True)
y_train = y_train_full.iloc[:train_size].reset_index(drop=True)
X_test = X_test_full.iloc[:test_size].reset_index(drop=True)
y_test = y_test_full.iloc[:test_size].reset_index(drop=True)
X_holdout = X_holdout.iloc[:holdout_size].reset_index(drop=True)
y_holdout = y_holdout.iloc[:holdout_size].reset_index(drop=True)

print("\nDataset Partitioning Complete:")
print(f"  Train (IN):    {len(X_train):,} samples - Used for training")
print(f"  Test (IN):     {len(X_test):,} samples - Seen during training distribution")
print(f"  Holdout (OUT): {len(X_holdout):,} samples - NEVER seen by model")
print("=" * 50)

print("\nℹ️  Key Concept:")
print("  - IN samples: Part of training distribution (train + test)")
print("  - OUT samples: Completely separate (holdout)")
print("  - Privacy attacks exploit behavioral differences between IN and OUT")
print("\n✓ Dataset partitioned for privacy experiments!")

---

## **Step 2: Train Victim Model (Intentionally Overfitted)** 🎯

### **What We're Doing**
Training a neural network that **intentionally overfits** to amplify privacy leakage signals. In real-world scenarios, overfitting happens unintentionally, but we'll force it here for clear demonstration.

### **Why Overfitting Matters for Privacy**
- **Memorization**: Overfitted models remember training samples more precisely
- **Confidence gap**: Higher confidence on IN samples (seen during training) vs OUT samples
- **Privacy leakage**: This gap enables membership inference and other attacks

### **How We Overfit**
1. **High capacity model**: Large hidden layers (256→128 neurons)
2. **Many training epochs**: 50 iterations (vs typical 20-30)
3. **No early stopping**: Let model fully memorize training data
4. **No regularization**: No dropout, no L2 penalty

### **Security Implication**
In production, overfitting is a **privacy vulnerability**:
- Recommendation systems that memorize user behavior
- Medical models that remember patient cases
- Financial models that memorize transaction patterns

### **What to Expect**
- Train accuracy: ~99% (high memorization)
- Test accuracy: ~92-95% (still generalizes reasonably)
- Holdout accuracy: ~92-95% (similar to test)
- **Overfit gap** (train - test): ~4-7% (indicates memorization)

In [ ]:
# Train an overfitted model (high capacity, many epochs, no regularization)
print("Training victim model (intentionally overfitted for privacy demonstration)...")
print("Architecture: 784 → 256 → 128 → 10")
print("(This may take 30-60 seconds...)\n")

victim_model = MLPClassifier(
    hidden_layer_sizes=(256, 128),  # Large capacity (encourages memorization)
    activation='relu',
    max_iter=50,                     # Many epochs (vs typical 20-30)
    random_state=42,
    verbose=False,
    early_stopping=False             # DISABLE early stopping (let it overfit!)
)

victim_model.fit(X_train, y_train)
print("✓ Training complete!")

# Evaluate on different data partitions
train_acc = victim_model.score(X_train, y_train)      # IN (training)
test_acc = victim_model.score(X_test, y_test)        # IN (test, but same distribution)
holdout_acc = victim_model.score(X_holdout, y_holdout)  # OUT (never seen)

print("\n" + "=" * 50)
print("VICTIM MODEL PERFORMANCE")
print("=" * 50)
print(f"Train (IN):    {train_acc:.4f} ({train_acc*100:.1f}%)")
print(f"Test (IN):     {test_acc:.4f} ({test_acc*100:.1f}%)")
print(f"Holdout (OUT): {holdout_acc:.4f} ({holdout_acc*100:.1f}%)")
print("=" * 50)

# Calculate overfit gap
overfit_gap = train_acc - test_acc
print(f"\n⚠️  Overfit Gap (Train - Test): {overfit_gap:.4f} ({overfit_gap*100:.1f}%)")

if overfit_gap > 0.03:
    print(f"    → Significant overfitting detected! Model memorizes training data.")
    print(f"    → This creates privacy vulnerabilities (membership inference, etc.)")
else:
    print(f"    → Limited overfitting. Privacy attacks may be less effective.")

print("\nℹ️  This gap indicates memorization, which enables privacy attacks!")
print("   Real-world models often overfit unintentionally, creating privacy risks.")

---

## **Step 3: Membership Inference Attack - Confidence-Based** 🔍🎯

### **What We're Doing**
Implementing the most common privacy attack: **membership inference**. Goal: Determine if a specific data point was in the training set, using only model outputs (black-box access).

### **Attack Hypothesis**
Overfitted models produce **different behavioral patterns** for IN vs OUT samples:
- **IN samples** (training data): Higher confidence, lower entropy, better accuracy
- **OUT samples** (unseen data): Lower confidence, higher entropy, more errors

### **Why This Matters**
Real-world implications:
- **Healthcare**: "Was patient X's record used to train this diagnostic model?"
- **Finance**: "Was my loan application in the training data?"
- **GDPR**: "Did the company delete my data or keep using it for training?"
- **Surveillance**: "Is my face in this recognition system's training set?"

### **Attack Mechanics**
1. Extract **confidence features** from model predictions
   - Max confidence (highest probability)
   - True class confidence (probability on correct label)
   - Entropy (measure of uncertainty)
   - Correctness (right/wrong prediction)
2. Train an **attack classifier** to distinguish IN vs OUT based on these features
3. Measure attack success rate (accuracy, AUC-ROC)

### **What We'll Measure**
- Attack accuracy (baseline: 50% random guess)
- AUC-ROC (area under ROC curve, measures discrimination ability)
- Feature importance (which signals matter most?)

In [ ]:
# Extract confidence features for membership inference attack
def get_confidence_features(model, X, y):
    """
    Extract privacy-relevant features from model predictions.
    
    These features capture how the model "feels" about samples:
    - Higher confidence + lower entropy = likely training sample (IN)
    - Lower confidence + higher entropy = likely unseen sample (OUT)
    
    Args:
        model: Trained classifier
        X: Input samples
        y: True labels
    
    Returns:
        DataFrame with confidence features
    """
    probs = model.predict_proba(X)  # Get probability distributions
    preds = model.predict(X)        # Get predicted labels
    
    features = []
    for i in range(len(X)):
        true_label = y.iloc[i] if hasattr(y, 'iloc') else y[i]
        pred_label = preds[i]
        
        # Feature 1: Max confidence (highest probability among all classes)
        max_conf = probs[i].max()
        
        # Feature 2: Confidence on TRUE class (probability of correct label)
        true_conf = probs[i][true_label]
        
        # Feature 3: Entropy (measure of uncertainty, lower = more confident)
        # Entropy = -Σ p(x) log p(x)
        entropy = -np.sum(probs[i] * np.log(probs[i] + 1e-10))
        
        # Feature 4: Correctness (binary: correct or wrong prediction)
        correct = 1 if pred_label == true_label else 0
        
        features.append({
            'max_confidence': max_conf,
            'true_confidence': true_conf,
            'entropy': entropy,
            'correct': correct
        })
    
    return pd.DataFrame(features)

print("Extracting confidence features for membership inference...")

# Get features for IN samples (test set - part of training distribution)
in_features = get_confidence_features(victim_model, X_test, y_test)
in_features['member'] = 1  # Label as IN (member)

# Get features for OUT samples (holdout - never seen by model)
out_features = get_confidence_features(victim_model, X_holdout, y_holdout)
out_features['member'] = 0  # Label as OUT (non-member)

# Combine into membership inference dataset
membership_data = pd.concat([in_features, out_features], ignore_index=True)

print("\n" + "=" * 50)
print("MEMBERSHIP INFERENCE DATASET")
print("=" * 50)
print(f"IN samples (member=1):  {len(in_features):,}")
print(f"OUT samples (member=0): {len(out_features):,}")
print(f"Total samples:          {len(membership_data):,}")
print("=" * 50)

print("\nSample features (first 10 rows):")
print(membership_data.head(10).to_string(index=False))
print("\n✓ Confidence features extracted!")

In [ ]:
# Analyze confidence differences between IN and OUT samples
print("Comparing IN vs OUT confidence statistics...\n")

print("=" * 60)
print("STATISTICAL COMPARISON: IN (member=1) vs OUT (member=0)")
print("=" * 60)
comparison = membership_data.groupby('member')[['max_confidence', 'true_confidence', 'entropy', 'correct']].mean()
comparison.index = ['OUT (non-member)', 'IN (member)']
print(comparison)
print("=" * 60)

print("\nℹ️  Expected pattern if model overfits:")
print("  - IN samples: Higher confidence, lower entropy, better accuracy")
print("  - OUT samples: Lower confidence, higher entropy, more errors\n")

# Visualize distributions to see privacy leakage
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Max confidence distribution
axes[0, 0].hist(in_features['max_confidence'], bins=30, alpha=0.7, label='IN (member)', color='blue', edgecolor='black')
axes[0, 0].hist(out_features['max_confidence'], bins=30, alpha=0.7, label='OUT (non-member)', color='red', edgecolor='black')
axes[0, 0].set_xlabel('Max Confidence', fontsize=11, fontweight='bold')
axes[0, 0].set_ylabel('Frequency', fontsize=11, fontweight='bold')
axes[0, 0].set_title('Max Confidence Distribution', fontsize=12, fontweight='bold')
axes[0, 0].legend(fontsize=10)
axes[0, 0].grid(alpha=0.3)

# Plot 2: True class confidence
axes[0, 1].hist(in_features['true_confidence'], bins=30, alpha=0.7, label='IN (member)', color='blue', edgecolor='black')
axes[0, 1].hist(out_features['true_confidence'], bins=30, alpha=0.7, label='OUT (non-member)', color='red', edgecolor='black')
axes[0, 1].set_xlabel('True Class Confidence', fontsize=11, fontweight='bold')
axes[0, 1].set_ylabel('Frequency', fontsize=11, fontweight='bold')
axes[0, 1].set_title('Confidence on True Class', fontsize=12, fontweight='bold')
axes[0, 1].legend(fontsize=10)
axes[0, 1].grid(alpha=0.3)

# Plot 3: Entropy distribution
axes[1, 0].hist(in_features['entropy'], bins=30, alpha=0.7, label='IN (member)', color='blue', edgecolor='black')
axes[1, 0].hist(out_features['entropy'], bins=30, alpha=0.7, label='OUT (non-member)', color='red', edgecolor='black')
axes[1, 0].set_xlabel('Entropy (Uncertainty)', fontsize=11, fontweight='bold')
axes[1, 0].set_ylabel('Frequency', fontsize=11, fontweight='bold')
axes[1, 0].set_title('Prediction Entropy Distribution', fontsize=12, fontweight='bold')
axes[1, 0].legend(fontsize=10)
axes[1, 0].grid(alpha=0.3)

# Plot 4: Correctness rate
correctness = membership_data.groupby('member')['correct'].mean()
axes[1, 1].bar(['OUT (non-member)', 'IN (member)'], [correctness[0], correctness[1]], 
               color=['red', 'blue'], alpha=0.7, edgecolor='black', linewidth=1.5)
axes[1, 1].set_ylabel('Prediction Accuracy', fontsize=11, fontweight='bold')
axes[1, 1].set_title('Prediction Accuracy by Membership', fontsize=12, fontweight='bold')
axes[1, 1].set_ylim([0, 1])
axes[1, 1].grid(alpha=0.3, axis='y')
for i, v in enumerate([correctness[0], correctness[1]]):
    axes[1, 1].text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

plt.suptitle('Privacy Leakage Visualization: IN vs OUT Sample Behavior', fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

print("\n⚠️  Key Observation:")
print("   If distributions differ significantly, membership inference attack will succeed!")
print("   Overlapping distributions = privacy-preserving model")

In [ ]:
# Train a membership inference attack classifier
print("Training membership inference attack model...")
print("(Using Random Forest to classify IN vs OUT based on confidence features)\n")

# Features for attack (what the attacker observes)
attack_features = ['max_confidence', 'true_confidence', 'entropy', 'correct']
X_attack = membership_data[attack_features]
y_attack = membership_data['member']

# Split into train/test for attack evaluation
X_attack_train, X_attack_test, y_attack_train, y_attack_test = train_test_split(
    X_attack, y_attack, 
    test_size=0.3,      # 30% for testing attack
    random_state=42, 
    stratify=y_attack
)

# Train attack model (Random Forest classifier)
attack_model = RandomForestClassifier(
    n_estimators=100,   # Ensemble of 100 decision trees
    random_state=42
)
attack_model.fit(X_attack_train, y_attack_train)
print("✓ Attack model trained!")

# Evaluate attack effectiveness
y_attack_pred = attack_model.predict(X_attack_test)
y_attack_proba = attack_model.predict_proba(X_attack_test)[:, 1]

attack_acc = accuracy_score(y_attack_test, y_attack_pred)
attack_auc = roc_auc_score(y_attack_test, y_attack_proba)

print("\n" + "=" * 60)
print("MEMBERSHIP INFERENCE ATTACK RESULTS")
print("=" * 60)
print(f"Attack Accuracy:  {attack_acc:.4f} ({attack_acc*100:.1f}%)")
print(f"Attack AUC-ROC:   {attack_auc:.4f}")
print(f"\nBaseline (random guess): 0.5000 (50%)")
print(f"Attack advantage over random: +{(attack_acc - 0.5)*100:.1f}%")
print("=" * 60)

# Interpret results
if attack_acc > 0.65:
    risk_level = "HIGH"
    print(f"\n⚠️  {risk_level} PRIVACY LEAKAGE DETECTED!")
    print(f"   Model significantly memorizes training data.")
    print(f"   Attacker can reliably infer membership with {attack_acc*100:.0f}% accuracy.")
elif attack_acc > 0.55:
    risk_level = "MEDIUM"
    print(f"\nℹ️  {risk_level} privacy leakage.")
    print(f"   Some memorization present. Attack better than random chance.")
else:
    risk_level = "LOW"
    print(f"\n✓ {risk_level} privacy leakage.")
    print(f"   Model generalizes well. Attack barely better than guessing.")

# Feature importance (which signals matter most for attack?)
feature_importance = pd.DataFrame({
    'Feature': attack_features,
    'Importance': attack_model.feature_importances_
}).sort_values('Importance', ascending=False)

print(f"\n" + "=" * 60)
print("ATTACK FEATURE IMPORTANCE")
print("=" * 60)
print(feature_importance.to_string(index=False))
print("=" * 60)
print("\nℹ️  Interpretation:")
print("   Higher importance = more useful for distinguishing IN vs OUT")
print("   Top features reveal which model behaviors leak privacy most")

---

## **Step 4: Model Inversion Attack - Feature Reconstruction** 🔄🖼️

### **What We're Doing**
Attempting to **reconstruct what training data looks like** by optimizing synthetic inputs to maximize model confidence on specific classes. This is a gradient-based privacy attack.

### **Attack Mechanics**
1. Start with random noise as input
2. Iteratively adjust input to maximize P(target_class | input)
3. Use gradient ascent: move in direction that increases class probability
4. Result: Synthetic sample that represents the "average" training example for that class

### **Why This Matters**
Real-world risks:
- **Face Recognition**: Reconstruct facial features from model trained on celebrity/employee faces
- **Medical Imaging**: Recreate patient scans from diagnostic models
- **Handwriting**: Reconstruct signature samples from fraud detection models
- **Proprietary Data**: Infer characteristics of private training datasets

### **The Math** (Simplified)
```
Goal: Find x that maximizes P(target_class | x)

For iterations:
    gradient = ∇_x P(target_class | x)  # How to change x to increase probability
    x = x + α × gradient                # Step in gradient direction
    x = clip(x, 0, 1)                   # Keep valid pixel range
```

### **What to Expect**
- Reconstructed images will show **class-characteristic features** (edges, shapes typical of that digit)
- Won't be exact copies of training samples (we're averaging across many examples)
- For specialized models (faces, medical), reconstruction can be much more precise

In [ ]:
# Implement model inversion via gradient ascent on input
def model_inversion_attack(model, target_class, num_iterations=500, learning_rate=0.1, reg_lambda=0.01):
    """
    Reconstruct a representative sample for target_class by optimizing input.
    
    Uses numerical gradient approximation to maximize P(target_class | x).
    
    Args:
        model: Victim classifier
        target_class: Class to reconstruct (0-9 for digits)
        num_iterations: Number of optimization steps
        learning_rate: Step size (alpha)
        reg_lambda: L2 regularization to prevent overfitting noise
    
    Returns:
        Reconstructed image (784-dim vector)
    """
    # Initialize with random noise (uniform distribution)
    x_inv = np.random.uniform(0, 1, 784)
    
    for iteration in range(num_iterations):
        # Step 1: Compute current confidence on target class
        probs = model.predict_proba([x_inv])[0]
        base_score = probs[target_class]
        
        # Step 2: Approximate gradient via finite differences
        # (In production, use automatic differentiation with PyTorch/TensorFlow)
        gradient = np.zeros_like(x_inv)
        epsilon = 1e-3
        
        # Sample 50 random pixels per iteration (faster than all 784)
        for _ in range(50):
            idx = np.random.randint(0, len(x_inv))
            x_perturbed = x_inv.copy()
            x_perturbed[idx] += epsilon
            
            probs_perturbed = model.predict_proba([x_perturbed])[0]
            perturbed_score = probs_perturbed[target_class]
            
            # Gradient = change in score / change in pixel
            gradient[idx] = (perturbed_score - base_score) / epsilon
        
        # Step 3: Gradient ascent (maximize score) with L2 regularization
        x_inv = x_inv + learning_rate * gradient - reg_lambda * x_inv
        
        # Step 4: Clip to valid pixel range [0, 1]
        x_inv = np.clip(x_inv, 0, 1)
        
        # Progress update
        if (iteration + 1) % 100 == 0:
            current_conf = model.predict_proba([x_inv])[0][target_class]
            print(f"  Iteration {iteration+1}/{num_iterations}: Confidence = {current_conf:.4f}")
    
    return x_inv

print("=" * 60)
print("LAUNCHING MODEL INVERSION ATTACK")
print("=" * 60)
print("Goal: Reconstruct representative samples for each digit class (0-9)")
print("Method: Gradient ascent to maximize class confidence")
print("(This may take 2-3 minutes...)\n")

inverted_samples = {}
for digit in range(10):
    print(f"Reconstructing digit {digit}...")
    inverted_samples[digit] = model_inversion_attack(
        victim_model, 
        digit, 
        num_iterations=500,
        learning_rate=0.1,
        reg_lambda=0.01
    )
    print()

print("=" * 60)
print("✓ Model inversion complete for all digits!")
print("=" * 60)

In [ ]:
# Visualize inverted samples vs real training samples
print("Visualizing model inversion results...\n")

fig, axes = plt.subplots(2, 10, figsize=(16, 4))

for digit in range(10):
    # Row 1: Real sample from training set
    real_idx = np.where(y_train == digit)[0][0]
    real_sample = X_train.iloc[real_idx].values.reshape(28, 28)
    
    axes[0, digit].imshow(real_sample, cmap='gray')
    axes[0, digit].set_title(f"Real {digit}", fontweight='bold')
    axes[0, digit].axis('off')
    
    # Row 2: Inverted sample (reconstructed by attack)
    inverted_sample = inverted_samples[digit].reshape(28, 28)
    axes[1, digit].imshow(inverted_sample, cmap='gray')
    
    # Get confidence on inverted sample
    inv_conf = victim_model.predict_proba([inverted_samples[digit]])[0][digit]
    inv_pred = victim_model.predict([inverted_samples[digit]])[0]
    
    axes[1, digit].set_title(f"Inv {digit}\nConf:{inv_conf:.2f}", fontweight='bold')
    axes[1, digit].axis('off')

axes[0, 0].set_ylabel('Real\nTraining', fontsize=13, fontweight='bold')
axes[1, 0].set_ylabel('Inverted\nReconstructed', fontsize=13, fontweight='bold')

plt.suptitle('Model Inversion Attack: Reconstructed vs Real Samples', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("=" * 60)
print("OBSERVATIONS")
print("=" * 60)
print("✓ Inverted samples show class-characteristic features")
print("  (e.g., curved shapes for 0, vertical lines for 1, loops for 8)")
print("\n✓ Not exact copies but 'average' representations")
print("  (model captures general features, not specific training examples)")
print("\n⚠️  For specialized models, reconstruction is more precise:")
print("  - Face recognition: Reconstruct facial features")
print("  - Medical imaging: Recover patient scan characteristics")
print("  - Proprietary data: Infer sensitive training data properties")
print("=" * 60)

---

## **Step 5: Property Inference Attack** 📊🔍

### **What We're Doing**
Inferring **hidden dataset properties** (e.g., class distribution, demographics) by analyzing model behavior, without access to training data.

### **Attack Scenario**
An attacker observes model predictions on uniform test data and infers:
- Class imbalance in training (e.g., "This fraud detector was trained on 90% legitimate, 10% fraud")
- Demographics (e.g., "This hiring model was trained mostly on male candidates")
- Sensitive attributes (e.g., "Medical model trained on predominantly elderly patients")

### **Why This Matters**
Property inference reveals:
- **Discrimination**: Biased training data (lawsuit risk, fairness concerns)
- **Privacy**: Dataset composition leaks sensitive demographics
- **Compliance**: GDPR/HIPAA violations if sensitive attributes are inferred
- **Competitive Intelligence**: Reveals competitor's data sources

### **Attack Mechanics**
1. Create victim model trained on **biased dataset** (simulating real-world imbalance)
2. Attacker queries model with **uniform test samples**
3. Analyze prediction bias and confidence patterns
4. Infer training class distribution

### **Real-World Example**
- Training data: 80% digit 0, 20% digit 9 (imbalanced)
- Model behavior: Predicts "0" more often, higher confidence on "0"
- Attacker infers: "Training data was imbalanced toward 0"

In [ ]:
# ========================================
# CREATE INTENTIONALLY BIASED TRAINING SET
# ========================================
# Property Inference exploits hidden dataset properties.
# Here, we simulate a real-world scenario where the training data
# is biased—contains more samples from certain classes.

print("=" * 60)
print("🎯 CREATING BIASED TRAINING SET")
print("=" * 60)

# Create biased subset: Include MORE samples from classes 0-4 (overrepresented digits)
# and FEWER samples from classes 5-9 (underrepresented digits)
# This simulates real-world bias (e.g., demographic imbalances in medical datasets)

biased_indices = []
for label in range(10):
    # Get all indices for this digit class (convert to numpy for np.where)
    y_train_np = y_train.to_numpy()
    class_positions = np.where(y_train_np == label)[0]
    
    if label < 5:
        # Classes 0-4: Include 80% of samples (OVERREPRESENTED)
        n_samples = int(len(class_positions) * 0.8)
    else:
        # Classes 5-9: Include only 20% of samples (UNDERREPRESENTED)
        n_samples = int(len(class_positions) * 0.2)
    
    # Randomly select the specified number of samples
    selected = np.random.choice(class_positions, n_samples, replace=False)
    biased_indices.extend(selected)

# Create the biased training set using iloc for pandas DataFrames
X_biased = X_train.iloc[biased_indices].reset_index(drop=True)
y_biased = y_train.iloc[biased_indices].reset_index(drop=True)

# ========================================
# TRAIN MODEL ON BIASED DATA
# ========================================
# The attacker doesn't know about this bias—they only query the model
# Goal: Can the attacker infer the hidden class distribution?

print("\n🔨 Training Model on Biased Dataset...")
biased_model = MLPClassifier(hidden_layer_sizes=(64,), max_iter=10, random_state=42, verbose=False)
biased_model.fit(X_biased, y_biased)
print("✅ Biased model trained successfully!")
print("=" * 60)

In [ ]:
# ========================================
# PROPERTY INFERENCE: INFER CLASS DISTRIBUTION
# ========================================
# Attacker Strategy:
# 1. Query the model with many test samples from each class
# 2. Measure prediction confidence for each class
# 3. Hypothesis: The model is more confident on classes it saw more during training

print("=" * 60)
print("🔍 INFERRING HIDDEN CLASS DISTRIBUTION")
print("=" * 60)

# Query the model with test samples and measure average confidence per class
inferred_distribution = {}

for label in range(10):
    # Get test samples for this class (attacker uses publicly available test data)
    y_test_np = y_test.to_numpy()
    class_test_indices = np.where(y_test_np == label)[0][:100]  # Use 100 samples per class
    X_class_test = X_test.iloc[class_test_indices]
    
    # Get model predictions and confidence scores
    probabilities = biased_model.predict_proba(X_class_test)
    
    # Calculate average confidence when the model predicts this class
    # Higher confidence suggests the model saw more samples of this class during training
    avg_confidence = np.mean(np.max(probabilities, axis=1))
    
    inferred_distribution[label] = avg_confidence

# ========================================
# COMPARE INFERRED VS ACTUAL DISTRIBUTION
# ========================================

# Calculate actual class distribution in biased training set
actual_distribution = {}
y_biased_np = y_biased.to_numpy()
for label in range(10):
    count = np.sum(y_biased_np == label)
    actual_distribution[label] = count / len(y_biased)

print("\n📊 Results:")
print("-" * 60)
print(f"{'Class':<8} {'Actual %':<12} {'Inferred Conf.':<18} {'Status'}")
print("-" * 60)

for label in range(10):
    actual_pct = actual_distribution[label] * 100
    inferred_conf = inferred_distribution[label]
    
    # Determine if class was overrepresented or underrepresented
    status = "🔴 Underrep." if label >= 5 else "🟢 Overrep."
    
    print(f"{label:<8} {actual_pct:<12.2f} {inferred_conf:<18.4f} {status}")

print("=" * 60)
print("\n📋 OBSERVATIONS:")
print("=" * 60)
print("""
1. **Attack Success**: Inferred confidence correlates with actual class distribution
   - Classes 0-4 (overrepresented): Higher confidence scores
   - Classes 5-9 (underrepresented): Lower confidence scores

2. **Privacy Leak**: Attacker learned hidden training data properties without access
   - No direct access to training data required
   - Only black-box queries to model needed

3. **Real-World Implications**
   - Healthcare: Infer demographic composition (age, gender, ethnicity) of patient data
   - Finance: Determine income distribution in credit scoring datasets
   - Hiring: Detect bias in applicant pool composition

4. **Why This Matters**
   - Violates data privacy regulations (GDPR Article 22: right to explanation)
   - Reveals potentially sensitive dataset statistics
   - Can expose discriminatory training practices
""")
print("=" * 60)

---

## **Step 6: Model Extraction Attack** 🎯💾

### **What We're Doing**
Demonstrate **model extraction** (model stealing)—replicating a victim model's functionality by querying it as a black box.

### **Attack Scenario**
- **Victim**: Company with proprietary ML model (e.g., fraud detection API)
- **Attacker**: Competitor who wants to steal model without paying licensing fees
- **Method**: Query the victim model with carefully chosen inputs, use predictions as training labels

### **How It Works**

**Step 1: Query Budget**
- Attacker sends queries to victim model API
- Limited by rate limits, cost, or detection risk
- Typically 1K - 100K queries

**Step 2: Training Data Selection**
- **Random sampling**: Query with random inputs
- **Active learning**: Intelligently select informative queries (advanced)

**Step 3: Train Surrogate Model**
- Use victim's predictions as ground-truth labels
- Train attacker's model to mimic victim's behavior
- Goal: Match victim's accuracy and decision boundaries

### **Success Metrics**
- **Agreement**: % of inputs where surrogate matches victim
- **Accuracy**: Surrogate's performance on true test set
- **Fidelity**: How well surrogate replicates victim's decision function

### **Real-World Examples**
- 🏦 **Financial Fraud Detection**: Steal proprietary fraud models via API queries
- 🖼️ **Face Recognition**: Replicate commercial face recognition APIs (e.g., Microsoft Azure, AWS Rekognition)
- 🩺 **Medical Diagnosis**: Extract clinical decision support models

### **Why This Matters**
- **IP Theft**: Years of R&D stolen with black-box queries
- **Revenue Loss**: Bypass licensing fees for commercial ML APIs
- **Security Risk**: Attacker uses surrogate to craft adversarial attacks offline

In [ ]:
# ========================================
# MODEL EXTRACTION ATTACK SIMULATION
# ========================================
# Attacker goal: Replicate the victim model's functionality
# Method: Query victim model, use predictions to train surrogate

print("=" * 60)
print("🎯 MODEL EXTRACTION ATTACK")
print("=" * 60)

# ========================================
# STEP 1: GENERATE QUERY SAMPLES
# ========================================
# In real attacks, attacker uses:
# - Random samples from input domain
# - Public datasets (e.g., ImageNet for vision models)
# - Synthetic data generation
# Here, we use test set (simulating publicly available data)

# Clamp query budget to available test set size
requested_queries = 5000  # Query budget (limited by cost/rate limits)
n_available = len(X_test)
n_queries = int(min(requested_queries, n_available))

# Reproducible sampling without replacement
rng = np.random.default_rng(42)
query_indices = rng.choice(n_available, n_queries, replace=False)
X_queries = X_test.iloc[query_indices]

print(f"\n📡 Querying Victim Model...")
print(f"   Total Queries: {n_queries}")

# Query the victim model (get predictions, not true labels)
# In real scenario, this is done via API calls (e.g., REST requests)
stolen_labels = victim_model.predict(X_queries)

print(f"✅ Received {len(stolen_labels)} predictions from victim model")

# ========================================
# STEP 2: TRAIN SURROGATE MODEL
# ========================================
# Use victim's predictions as training labels
# Attacker's model learns to mimic victim's behavior

print(f"\n🔨 Training Surrogate Model...")
print(f"   Using victim's predictions as ground truth")

surrogate_model = MLPClassifier(hidden_layer_sizes=(64,), max_iter=10, random_state=42, verbose=False)
surrogate_model.fit(X_queries, stolen_labels)

print("✅ Surrogate model trained successfully!")
print("=" * 60)


In [ ]:
# ========================================
# EVALUATE MODEL EXTRACTION SUCCESS
# ========================================
# Compare surrogate model against victim model

print("=" * 60)
print("📊 MODEL EXTRACTION EVALUATION")
print("=" * 60)

# Use remaining test set (not used for queries) for evaluation
eval_indices = [i for i in range(len(X_test)) if i not in query_indices]

# If no remaining samples, fallback to a holdout subset
if len(eval_indices) == 0:
    # Leave at least 20% of test set for evaluation
    n_eval = max(1, int(0.2 * len(X_test)))
    rng = np.random.default_rng(123)
    eval_indices = rng.choice(len(X_test), n_eval, replace=False)

X_eval = X_test.iloc[eval_indices]
y_eval = y_test.iloc[eval_indices]

# Guard against empty or wrong-shaped inputs
if len(X_eval) == 0:
    raise ValueError("Evaluation set is empty; reduce query budget or provide holdout samples.")

# Get predictions from both models
victim_predictions = victim_model.predict(X_eval)
surrogate_predictions = surrogate_model.predict(X_eval)

# ========================================
# METRIC 1: AGREEMENT (Fidelity)
# ========================================
# Percentage of inputs where surrogate matches victim's predictions
agreement = np.mean(victim_predictions == surrogate_predictions) * 100

# ========================================
# METRIC 2: ACCURACY ON TRUE LABELS
# ========================================
# How well each model performs on actual test data
victim_accuracy = accuracy_score(y_eval, victim_predictions) * 100
surrogate_accuracy = accuracy_score(y_eval, surrogate_predictions) * 100

print("\n📋 Results:")
print("-" * 60)
print(f"Metric                          Value")
print("-" * 60)
print(f"Agreement (Fidelity)            {agreement:.2f}%")
print(f"Victim Model Accuracy           {victim_accuracy:.2f}%")
print(f"Surrogate Model Accuracy        {surrogate_accuracy:.2f}%")
print(f"Accuracy Gap                    {abs(victim_accuracy - surrogate_accuracy):.2f}%")
print("=" * 60)

print("\n📋 INTERPRETATION:")
print("=" * 60)
print(f"""
**Agreement (Fidelity): {agreement:.1f}%**
- Measures how often surrogate makes SAME predictions as victim
- {agreement:.1f}% → Surrogate successfully replicates victim's behavior
- High fidelity = successful model theft

**Accuracy Comparison:**
- Victim: {victim_accuracy:.1f}% | Surrogate: {surrogate_accuracy:.1f}%
- Gap: {abs(victim_accuracy - surrogate_accuracy):.1f}% → {'Minimal' if abs(victim_accuracy - surrogate_accuracy) < 5 else 'Moderate' if abs(victim_accuracy - surrogate_accuracy) < 10 else 'Significant'} difference
- Surrogate achieves {'comparable' if abs(victim_accuracy - surrogate_accuracy) < 5 else 'reasonable' if abs(victim_accuracy - surrogate_accuracy) < 10 else 'lower'} performance
""")


---

## **Step 7: Defense - Output Perturbation and Rate Limiting** 🛡️🔒

### **What We're Doing**
Implementing **privacy-preserving prediction** mechanisms to reduce information leakage and prevent extraction attacks.

### **Defense Strategies**

**1. Output Perturbation (Noise Injection)**
- Add calibrated noise to confidence scores (Laplace/Gaussian)
- Reduces distinguishability between IN and OUT samples
- Mitigates membership inference attacks

**2. Top-K Filtering**
- Return only top-k classes (not full probability vector)
- Limits information available to attacker
- Reduces model extraction effectiveness

**3. Confidence Calibration**
- Clip extreme confidences (prevent overconfident predictions)
- Smooth probability distributions
- Reduces membership inference signals

### **Privacy-Utility Tradeoff**
- ✅ **More noise** → Better privacy (lower attack success)
- ⚠️ **More noise** → Lower accuracy (utility loss)
- **Goal**: Find optimal balance for your threat model

### **Real-World Deployment**
- **API rate limiting**: Prevent large-scale extraction attacks
- **Query monitoring**: Detect suspicious systematic sampling
- **Authentication**: Track who queries the model
- **Audit logging**: Record all predictions for investigation

In [ ]:
# Implement privacy-preserving prediction with output perturbation
def private_predict(model, X, noise_scale=0.1, top_k=3):
    """
    Privacy-preserving prediction with multiple defense mechanisms.
    
    Defenses implemented:
    1. Add Laplace noise to confidence scores (differential privacy-inspired)
    2. Return only top-k classes (limit information leakage)
    3. Clip extreme confidences (reduce overconfident predictions)
    
    Args:
        model: Trained classifier
        X: Input samples (pandas DataFrame or numpy array)
        noise_scale: Laplace noise parameter (higher = more privacy, less accuracy)
        top_k: Number of top classes to return (lower = more privacy)
    
    Returns:
        Noisy, sparse probability distributions
    """
    probs = model.predict_proba(X)
    
    # Defense 1: Add calibrated Laplace noise
    # Laplace(0, scale) adds differential privacy-like protection
    noise = np.random.laplace(0, noise_scale, probs.shape)
    noisy_probs = probs + noise
    
    # Defense 2: Clip to valid range [0, 1] and renormalize with epsilon to avoid divide-by-zero
    noisy_probs = np.clip(noisy_probs, 0, 1)
    row_sums = noisy_probs.sum(axis=1, keepdims=True)
    eps = 1e-12
    noisy_probs = noisy_probs / (row_sums + eps)
    
    # Ensure top_k is within valid bounds
    n_classes = noisy_probs.shape[1]
    k = int(max(1, min(top_k, n_classes)))
    
    # Defense 3: Return only top-k (reducing information leakage)
    top_k_indices = np.argsort(noisy_probs, axis=1)[:, -k:]
    
    # Create sparse output (zero out non-top-k classes)
    sparse_probs = np.zeros_like(noisy_probs)
    for i in range(len(noisy_probs)):
        sparse_probs[i, top_k_indices[i]] = noisy_probs[i, top_k_indices[i]]
    
    # Renormalize top-k to sum to 1 (with epsilon)
    sparse_row_sums = sparse_probs.sum(axis=1, keepdims=True)
    sparse_probs = sparse_probs / (sparse_row_sums + eps)
    
    return sparse_probs

print("=" * 60)
print("PRIVACY-PRESERVING PREDICTION DEMONSTRATION")
print("=" * 60)
print("Testing privacy-preserving prediction on sample inputs...\n")

# Compare original vs private predictions
sample_X = X_test.iloc[:5]
sample_y = y_test.iloc[:5]

original_probs = victim_model.predict_proba(sample_X)
private_probs = private_predict(victim_model, sample_X, noise_scale=0.1, top_k=3)

print("Original predictions (full probability vector, 10 classes):")
for i in range(len(sample_X)):
    print(f"Sample {i} (true={sample_y.iloc[i]}): {original_probs[i].round(3)}")

print("\nPrivate predictions (top-3 classes with noise):")
for i in range(len(sample_X)):
    print(f"Sample {i} (true={sample_y.iloc[i]}): {private_probs[i].round(3)}")

print("\n" + "=" * 60)
print("OBSERVATIONS")
print("=" * 60)
print("✓ Private predictions return only top-3 classes (7 classes zeroed out)")
print("✓ Noise added reduces precise confidence values")
print("✓ Attacker gets less information per query")
print("\n⚠️  Tradeoff: Privacy ↑, Information ↓, Potential accuracy impact")
print("=" * 60)


In [ ]:
# Evaluate privacy vs utility trade-off
print("\nEvaluating Privacy-Utility Trade-off...\n")

noise_levels = [0.0, 0.05, 0.1, 0.2, 0.3]
results = []

for noise in noise_levels:
    # Get private predictions
    private_probs = private_predict(victim_model, X_test, noise_scale=noise, top_k=3)
    private_preds = np.argmax(private_probs, axis=1)
    
    # Accuracy
    acc = accuracy_score(y_test, private_preds)
    
    # Membership inference resistance (re-test attack)
    in_features_private = get_confidence_features(
        victim_model,
        X_test.iloc[:200],
        y_test.iloc[:200]
    )
    
    # Add noise simulation to features
    in_features_private['max_confidence'] += np.random.laplace(0, noise, len(in_features_private))
    in_features_private['true_confidence'] += np.random.laplace(0, noise, len(in_features_private))
    in_features_private['max_confidence'] = in_features_private['max_confidence'].clip(0, 1)
    in_features_private['true_confidence'] = in_features_private['true_confidence'].clip(0, 1)
    in_features_private['member'] = 1
    
    out_features_private = get_confidence_features(
        victim_model,
        X_holdout.iloc[:200],
        y_holdout.iloc[:200]
    )
    out_features_private['max_confidence'] += np.random.laplace(0, noise, len(out_features_private))
    out_features_private['true_confidence'] += np.random.laplace(0, noise, len(out_features_private))
    out_features_private['max_confidence'] = out_features_private['max_confidence'].clip(0, 1)
    out_features_private['true_confidence'] = out_features_private['true_confidence'].clip(0, 1)
    out_features_private['member'] = 0
    
    combined_private = pd.concat([in_features_private, out_features_private], ignore_index=True)
    attack_acc_private = attack_model.score(
        combined_private[attack_features],
        combined_private['member']
    )
    
    results.append({
        'noise': noise,
        'accuracy': acc,
        'attack_success': attack_acc_private
    })

results_df = pd.DataFrame(results)
print("Privacy-Utility Trade-off:")
print(results_df)

# Visualize trade-off
fig, ax1 = plt.subplots(figsize=(10, 6))

ax1.plot(results_df['noise'], results_df['accuracy'], 'b-o', label='Model Accuracy', linewidth=2)
ax1.set_xlabel('Noise Scale', fontsize=12)
ax1.set_ylabel('Accuracy', color='b', fontsize=12)
ax1.tick_params(axis='y', labelcolor='b')
ax1.set_ylim([0.7, 1.0])

ax2 = ax1.twinx()
ax2.plot(results_df['noise'], results_df['attack_success'], 'r-s', label='Attack Success', linewidth=2)
ax2.set_ylabel('Membership Attack Success', color='r', fontsize=12)
ax2.tick_params(axis='y', labelcolor='r')
ax2.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='Random Guess')
ax2.set_ylim([0.4, 0.8])

plt.title('Privacy-Utility Trade-off: Output Noise vs Accuracy and Attack Resistance', fontsize=14)
fig.tight_layout()
plt.show()

print("\nKey Insight: Increased noise improves privacy (reduces attack success) but may reduce accuracy.")

---

## **Step 8: Defense - Differential Privacy Overview** 🔐📊

### **What We're Doing**
Introduction to **Differential Privacy (DP)** concepts—the gold standard for formal privacy guarantees in machine learning.

### **What is Differential Privacy?**
Differential Privacy provides **mathematical guarantees** that:
> "Including or excluding any single individual's data doesn't significantly change model outputs."

### **Key Parameters**

**ε (epsilon) - Privacy Budget**
- Lower ε = Stronger privacy
- Higher ε = Weaker privacy
- Typical values:
  - ε < 1: Strong privacy (recommended for sensitive data)
  - ε = 1-10: Moderate privacy
  - ε > 10: Weak privacy

**δ (delta) - Privacy Failure Probability**
- Probability of privacy guarantee breaking down
- Typically very small (e.g., 10⁻⁵)

### **Differential Privacy in Machine Learning (DP-SGD)**

**Mechanisms:**
1. **Gradient Clipping**: Bound each training sample's influence on gradients
2. **Noise Injection**: Add calibrated Gaussian noise to gradients during training
3. **Privacy Accounting**: Track cumulative privacy loss across epochs

**Formula (Simplified):**
```
For each training step:
    1. Clip per-sample gradients: g_clipped = clip(g, max_norm)
    2. Add noise: g_noisy = g_clipped + Gaussian(0, σ²)
    3. Update model: θ = θ - α × g_noisy
```

### **Production Implementation**
- **TensorFlow Privacy**: Google's DP library for TensorFlow
- **Opacus**: PyTorch library for differential privacy
- **Privacy accounting**: Moments accountant or Rényi DP

### **Tradeoff**
- ✅ **Pro**: Formal privacy guarantees (provable bounds)
- ✅ **Pro**: Protects against ALL privacy attacks (membership inference, inversion, etc.)
- ⚠️ **Con**: Reduced model accuracy (noise degrades learning)
- ⚠️ **Con**: Increased training time (per-sample gradient computation)

In [ ]:
# ========================================
# DIFFERENTIAL PRIVACY CONCEPTUAL DEMO
# ========================================
# Note: This is a simplified simulation to demonstrate DP concepts.
# Production DP requires libraries like TensorFlow Privacy or Opacus.

def gaussian_mechanism(true_value, sensitivity, epsilon, delta=1e-5):
    """
    Apply Gaussian mechanism for differential privacy.
    
    Parameters:
    -----------
    true_value : float
        The actual value to privatize (e.g., model prediction confidence)
    sensitivity : float
        Maximum possible change when adding/removing one record
        For confidence scores: typically 1.0 (range [0,1])
    epsilon : float
        Privacy budget (lower = more privacy, more noise)
    delta : float
        Privacy failure probability (typically very small, e.g., 1e-5)
    
    Returns:
    --------
    float : Privatized value with calibrated Gaussian noise
    
    How It Works:
    -------------
    σ² = 2 × ln(1.25/δ) × sensitivity² / epsilon²
    noisy_value = true_value + N(0, σ²)
    """
    import numpy as np
    
    # Calculate noise scale (sigma) based on epsilon and delta
    # Formula: σ = sensitivity × sqrt(2 × ln(1.25/δ)) / ε
    sigma = sensitivity * np.sqrt(2 * np.log(1.25 / delta)) / epsilon
    
    # Add Gaussian noise with calculated standard deviation
    noise = np.random.normal(0, sigma)
    noisy_value = true_value + noise
    
    return noisy_value

# ========================================
# SIMULATE DIFFERENTIAL PRIVACY IMPACT
# ========================================
print("=" * 60)
print("🔐 DIFFERENTIAL PRIVACY IMPACT DEMONSTRATION")
print("=" * 60)

# Original model prediction confidence for class 3
true_confidence = 0.92

print(f"\n📊 Original Prediction:")
print(f"   True Confidence for Class 3: {true_confidence:.4f}")
print()

# Test different privacy budgets (epsilon values)
epsilon_values = [0.1, 1.0, 5.0, 10.0]
sensitivity = 1.0  # Maximum change in confidence score (range [0, 1])

print("🧪 Testing Different Privacy Budgets (ε):")
print("-" * 60)

for eps in epsilon_values:
    # Apply DP mechanism 5 times to show variance
    noisy_predictions = [gaussian_mechanism(true_confidence, sensitivity, eps) 
                        for _ in range(5)]
    
    avg_noise = np.mean([abs(p - true_confidence) for p in noisy_predictions])
    
    print(f"\n   ε = {eps:5.1f} (Privacy: {'Strong' if eps < 1 else 'Moderate' if eps < 5 else 'Weak'})")
    print(f"   Sample Noisy Predictions: {[f'{p:.4f}' for p in noisy_predictions[:3]]}")
    print(f"   Average Noise Magnitude: ±{avg_noise:.4f}")

print("\n" + "=" * 60)
print("📋 OBSERVATIONS:")
print("=" * 60)
print("""
1. **Lower ε → More Noise → Stronger Privacy**
   - ε = 0.1: Large noise swamps signal, predictions unreliable
   - ε = 10.0: Minimal noise, predictions stay accurate

2. **Privacy-Utility Tradeoff**
   - Strong privacy (ε < 1): Protects membership, but reduces accuracy
   - Weak privacy (ε > 10): Better accuracy, but leaks information

3. **Production Recommendations**
   - Healthcare/Finance: ε = 0.5 - 1.0 (strong privacy required)
   - General applications: ε = 1.0 - 5.0 (balanced tradeoff)
   - Public datasets: ε > 5.0 (lower privacy concerns)

4. **Why This Matters**
   - DP provides *provable* privacy guarantees (not heuristics)
   - Gold standard for GDPR, CCPA compliance
   - Prevents ALL privacy attacks (membership, inversion, etc.)
""")
print("=" * 60)

---

## **Step 9: Summary - Privacy Attack Landscape** 📊🔍

### **What We Covered**
Comprehensive hands-on exploration of **four major privacy attack types** and their defenses in machine learning systems.

In [ ]:
# ========================================
# PRIVACY ATTACKS SUMMARY COMPARISON
# ========================================

import pandas as pd

print("=" * 80)
print("📊 PRIVACY ATTACK LANDSCAPE - COMPREHENSIVE SUMMARY")
print("=" * 80)

# Create comprehensive comparison table
attack_summary = pd.DataFrame({
    'Attack Type': [
        'Membership Inference',
        'Model Inversion',
        'Property Inference',
        'Model Extraction'
    ],
    'Target': [
        'Individual records',
        'Training data features',
        'Dataset statistics',
        'Model functionality'
    ],
    'What Attacker Learns': [
        'Was person X in training set?',
        'What did person X look like?',
        'Dataset demographics/bias',
        'How to replicate model'
    ],
    'Attack Complexity': [
        'Low',
        'Medium-High',
        'Low-Medium',
        'Low'
    ],
    'Query Budget': [
        '100-1000',
        '1000-10000',
        '1000-5000',
        '5000-100000'
    ],
    'Real-World Impact': [
        'GDPR violations, privacy breach',
        'Identity theft, face reconstruction',
        'Discrimination detection, bias exposure',
        'IP theft, revenue loss'
    ]
})

print("\n" + "=" * 80)
print(attack_summary.to_string(index=False))
print("=" * 80)

# ========================================
# DEFENSE STRATEGIES COMPARISON
# ========================================

print("\n\n📋 DEFENSE STRATEGIES COMPARISON")
print("=" * 80)

defense_summary = pd.DataFrame({
    'Defense Mechanism': [
        'Output Perturbation',
        'Top-K Filtering',
        'Confidence Calibration',
        'Differential Privacy',
        'Model Watermarking',
        'Query Rate Limiting'
    ],
    'Protects Against': [
        'Membership Inference',
        'Membership Inference, Model Inversion',
        'Membership Inference',
        'ALL privacy attacks',
        'Model Extraction',
        'Model Extraction'
    ],
    'Privacy Strength': [
        'Medium',
        'Medium-High',
        'Low-Medium',
        'High (provable)',
        'N/A (detection only)',
        'Low (deterrent)'
    ],
    'Accuracy Impact': [
        'Low (1-3%)',
        'Low-Medium (2-5%)',
        'Minimal (<1%)',
        'Medium-High (3-10%)',
        'Minimal (<1%)',
        'None'
    ],
    'Deployment Difficulty': [
        'Easy',
        'Easy',
        'Medium',
        'Hard',
        'Medium',
        'Easy'
    ]
})

print("\n" + "=" * 80)
print(defense_summary.to_string(index=False))
print("=" * 80)

# ========================================
# RISK ASSESSMENT MATRIX
# ========================================

print("\n\n🚨 PRIVACY RISK ASSESSMENT BY DOMAIN")
print("=" * 80)

risk_matrix = pd.DataFrame({
    'Domain': [
        'Healthcare (Patient Records)',
        'Finance (Credit Scoring)',
        'Face Recognition',
        'Autonomous Vehicles',
        'Social Media Recommendations'
    ],
    'Membership Inference': [
        '🔴 CRITICAL',
        '🔴 CRITICAL',
        '🟡 MODERATE',
        '🟢 LOW',
        '🟡 MODERATE'
    ],
    'Model Inversion': [
        '🟡 MODERATE',
        '🟢 LOW',
        '🔴 CRITICAL',
        '🟢 LOW',
        '🟡 MODERATE'
    ],
    'Property Inference': [
        '🔴 CRITICAL',
        '🔴 CRITICAL',
        '🟡 MODERATE',
        '🟢 LOW',
        '🟡 MODERATE'
    ],
    'Model Extraction': [
        '🟡 MODERATE',
        '🔴 CRITICAL',
        '🔴 CRITICAL',
        '🔴 CRITICAL',
        '🟡 MODERATE'
    ]
})

print("\n" + "=" * 80)
print(risk_matrix.to_string(index=False))
print("=" * 80)

print("\n\n📋 OBSERVATIONS:")
print("=" * 80)
print("""
1. **Attack Diversity**: Privacy attacks target different aspects
   - Membership: Individual privacy violations
   - Inversion: Feature reconstruction
   - Property: Statistical leakage
   - Extraction: Intellectual property theft

2. **Defense Tradeoffs**: No single defense solves all attacks
   - Differential Privacy: Strongest guarantees, highest accuracy cost
   - Output Perturbation: Balanced tradeoff, practical deployment
   - Rate Limiting: Minimal impact, limited effectiveness

3. **Domain-Specific Risks**: Privacy threats vary by application
   - Healthcare: Membership + Property Inference (HIPAA, GDPR)
   - Face Recognition: Model Inversion (biometric reconstruction)
   - Finance: Model Extraction (IP theft of fraud detection models)

4. **Regulatory Compliance**
   - GDPR Article 22: Right to explanation (affects all attacks)
   - HIPAA Privacy Rule: Protects patient membership information
   - CCPA: California Consumer Privacy Act (membership inference)
   - BIPA: Biometric Information Privacy Act (model inversion)
""")
print("=" * 80)

---

## **🎯 Key Takeaways: Privacy in Machine Learning**

### **1. Privacy Attacks are Real and Practical**
- ✅ **Demonstrated**: Four major attack types on standard ML models
- ⚠️ **Risk**: Attacks work with limited queries (100-10,000) and minimal technical knowledge
- 🚨 **Impact**: Privacy violations have legal, ethical, and financial consequences

---

### **2. Attack Surface Understanding**

**Membership Inference** 📍
- **Threat**: Determine if specific individual's data was used for training
- **Vulnerability**: Model overfitting → higher confidence on training samples
- **Real Example**: "Was patient John Doe in this cancer study dataset?"
- **Regulation**: GDPR Article 17 (right to be forgotten)

**Model Inversion** 🖼️
- **Threat**: Reconstruct training data features from model outputs
- **Vulnerability**: Model memorization of training patterns
- **Real Example**: Reconstruct faces from face recognition model predictions
- **Regulation**: BIPA (Biometric Information Privacy Act)

**Property Inference** 📊
- **Threat**: Infer hidden dataset statistics and biases
- **Vulnerability**: Model confidence patterns reflect training distribution
- **Real Example**: Detect demographic bias in hiring/lending datasets
- **Regulation**: EU AI Act (bias disclosure requirements)

**Model Extraction** 💾
- **Threat**: Steal proprietary model functionality via black-box queries
- **Vulnerability**: Unrestricted API access enables training surrogate models
- **Real Example**: Replicate commercial fraud detection APIs
- **Impact**: IP theft, revenue loss, security amplification

---

### **3. Defense Strategy Framework**

**Tiered Defense Approach**

**Layer 1: Output Perturbation (Easy, Low Impact)**
- Add Laplace noise to predictions
- Implement top-k filtering
- Calibrate confidence scores
- ✅ **Use when**: General production deployments
- ⚠️ **Limitation**: Heuristic, no formal guarantees

**Layer 2: Architectural Defenses (Medium Difficulty)**
- Regularization (L2, dropout) to prevent overfitting
- Early stopping to limit memorization
- Model ensembling to reduce individual model leakage
- ✅ **Use when**: Building privacy-aware models from scratch

**Layer 3: Differential Privacy (Hard, High Impact)**
- DP-SGD: Add calibrated noise during training
- Privacy accounting: Track cumulative privacy loss
- Formal guarantees: ε-δ privacy bounds
- ✅ **Use when**: Healthcare, finance, sensitive personal data
- ⚠️ **Cost**: 3-10% accuracy reduction

**Layer 4: Access Controls (Easy, Detection Focus)**
- Query rate limiting and monitoring
- CAPTCHA challenges for suspicious patterns
- Model watermarking to detect theft
- API authentication and audit logs
- ✅ **Use when**: Commercial ML APIs, high-value models

---

### **4. Practical Recommendations by Domain**

**🏥 Healthcare (Patient Records)**
- **Priority**: Membership Inference, Property Inference
- **Defense**: Differential Privacy (ε = 0.5-1.0) + Output Perturbation
- **Compliance**: HIPAA, GDPR Article 9 (special category data)
- **Tradeoff**: Accept 5-8% accuracy reduction for strong privacy

**💳 Finance (Credit/Fraud Models)**
- **Priority**: Model Extraction, Property Inference
- **Defense**: Rate limiting + API watermarking + Top-K filtering
- **Compliance**: FCRA (Fair Credit Reporting Act), GDPR Article 22
- **Tradeoff**: Monitor query patterns, limit API access

**👤 Face Recognition**
- **Priority**: Model Inversion, Model Extraction
- **Defense**: Differential Privacy + Confidence calibration
- **Compliance**: BIPA (Illinois), GDPR Article 9
- **Tradeoff**: Balance recognition accuracy with privacy guarantees

**🚗 Autonomous Vehicles**
- **Priority**: Model Extraction (IP protection)
- **Defense**: Model watermarking + Rate limiting
- **Compliance**: Trade secret protection
- **Tradeoff**: Minimal privacy risk, focus on IP protection

---

### **5. Regulatory Landscape**

**GDPR (General Data Protection Regulation)**
- Article 17: Right to erasure → Affects membership inference
- Article 22: Right to explanation → Affects all automated decisions
- Article 9: Special category data (health, biometrics) → Requires strong privacy

**CCPA (California Consumer Privacy Act)**
- Right to know what personal information is collected
- Right to deletion → Similar to GDPR Article 17

**HIPAA (Health Insurance Portability and Accountability Act)**
- Privacy Rule: Protects individually identifiable health information
- Membership inference = potential HIPAA violation

**BIPA (Biometric Information Privacy Act - Illinois)**
- Strict requirements for biometric data (faces, fingerprints)
- Model inversion attacks = direct BIPA violation

---

### **6. Research and Advanced Topics**

**Emerging Attacks**
- **Gradient Leakage**: Extract training data from federated learning gradients
- **Backdoor Model Extraction**: Steal not just functionality, but hidden backdoors
- **GAN Inversion**: Use generative models to reconstruct training images

**Advanced Defenses**
- **Secure Multi-Party Computation (SMPC)**: Train on encrypted data
- **Homomorphic Encryption**: Inference on encrypted inputs
- **Trusted Execution Environments (TEEs)**: Hardware-based privacy (Intel SGX)

**Open Research Questions**
- How to achieve differential privacy without accuracy loss?
- Can we detect model extraction in real-time?
- How to balance transparency (explainability) with privacy?

---

### **7. Hands-On Skills Developed** 💪

By completing this lab, you can now:
- ✅ **Implement** membership inference attacks using confidence features
- ✅ **Reconstruct** training data via model inversion (gradient ascent)
- ✅ **Detect** hidden dataset properties through property inference
- ✅ **Replicate** black-box models via model extraction
- ✅ **Deploy** privacy-preserving prediction mechanisms
- ✅ **Evaluate** privacy-utility tradeoffs quantitatively
- ✅ **Assess** domain-specific privacy risks

---

### **8. Next Steps for Security Analysts** 🚀

**Immediate Actions**
1. **Audit existing ML systems**: Identify privacy attack surfaces
2. **Implement basic defenses**: Start with output perturbation + rate limiting
3. **Monitor query patterns**: Detect potential extraction attempts
4. **Document privacy risks**: Create risk register for compliance

**Skill Development**
1. **Learn production DP libraries**: TensorFlow Privacy, Opacus
2. **Study federated learning**: Privacy-preserving distributed training
3. **Practice privacy auditing**: Use tools like ML Privacy Meter
4. **Stay updated**: Follow NeurIPS, ICML privacy workshops

**Resources**
- 📚 **Book**: "The Algorithmic Foundations of Differential Privacy" (Dwork & Roth)
- 🔧 **Tools**: TensorFlow Privacy, Opacus, ML Privacy Meter
- 📊 **Datasets**: MNIST, CIFAR-10 (for experimentation)
- 🌐 **Community**: PPML Workshop (Privacy Preserving Machine Learning)

---

### **🎓 Congratulations!**

You've completed a comprehensive exploration of **Privacy Attacks and Information Extraction** in machine learning systems. You now understand:
- How attackers extract sensitive information from ML models
- Why privacy attacks succeed (overfitting, memorization, unrestricted access)
- How to defend against privacy threats (DP, perturbation, rate limiting)
- Where privacy risks are highest (healthcare, finance, biometrics)
- What regulatory requirements apply (GDPR, HIPAA, BIPA)

**Remember**: Privacy is not a feature, it's a fundamental requirement for trustworthy AI systems.

---

### **📖 References and Further Reading**

1. Shokri et al. (2017): "Membership Inference Attacks Against Machine Learning Models"
2. Fredrikson et al. (2015): "Model Inversion Attacks that Exploit Confidence Information"
3. Tramèr et al. (2016): "Stealing Machine Learning Models via Prediction APIs"
4. Abadi et al. (2016): "Deep Learning with Differential Privacy"
5. Dwork & Roth (2014): "The Algorithmic Foundations of Differential Privacy"

---

**🔐 Stay Secure. Stay Private. 🔐**